# 05. Bias correction по in-situ бурениям

**Цель:** устранить cold bias, унаследованный от MODIS LST и TTOP-таргета.
Калибруем MAGT-предсказание на 33 чистых бурениях через k-fold CV,
выбираем минимальную стратегию (Occam), применяем к полной карте 2023.

**Входы:**
- `results/maps/FINAL_maps.npz` — предсказанная MAGT 2023
- `data/boreholes/clean_boreholes_33.csv` — 33 чистых бурения
- `results/maps/zones_2023.npz` — карта зон мерзлоты (опционально)

**Выходы:**
- `results/maps/MAGT_2023_bias_corrected.npz`
- `results/metrics/bias_correction_summary.csv`
- `results/figures/bias_correction_*.png`

**Acceptance:** RMSE против бурений упал, |bias| < 0.3°C, карта визуально не "перегрета".

In [ ]:
# Environment auto-detection
# Работает на Colab VM, локальном Jupyter, и Colab UI с local runtime
import os
from pathlib import Path

IN_COLAB_VM = (
    'COLAB_RELEASE_TAG' in os.environ or
    'COLAB_GPU' in os.environ
)

if IN_COLAB_VM:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE_DIR = Path('/content/drive/MyDrive/AI4Arctic')
    env_label = 'Colab VM'
else:
    # Local Jupyter (standalone ИЛИ Colab UI + local runtime)
    BASE_DIR = Path(os.environ.get(
        'AI4ARCTIC_HOME',
        Path.home() / 'Ai4Arctic'
    ))
    env_label = 'Local runtime'

print(f"Environment: {env_label}")
print(f"BASE_DIR: {BASE_DIR}")
assert BASE_DIR.exists(), f"BASE_DIR не найден: {BASE_DIR}. Установи AI4ARCTIC_HOME env var или скопируй проект в ~/Desktop/Ai4Arctic"

import sys
sys.path.insert(0, str(BASE_DIR))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import importlib
import src.bias_correction
importlib.reload(src.bias_correction)
from src.bias_correction import (
    load_borehole_data,
    pair_predictions_with_boreholes,
    assign_zone_by_magt,
    compare_strategies,
    pick_best_strategy,
    apply_correction,
    bootstrap_corrector,
    fit_global_linear,
)

MAPS_DIR = BASE_DIR / 'results' / 'maps'
METRICS_DIR = BASE_DIR / 'results' / 'metrics'
FIGURES_DIR = BASE_DIR / 'results' / 'figures'
BOREHOLES_CSV = BASE_DIR / 'data' / 'boreholes' / 'clean_boreholes_33.csv'

METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Загружаем данные

In [ ]:
final_maps = np.load(BASE_DIR / 'data' / 'maps_lh_v3.npz')
print("Ключи в FINAL_maps.npz:", list(final_maps.keys()))

# ВАЖНО: если у тебя другое имя ключа, замени ниже
# Финальное MAGT (с latent heat correction)
magt_2023 = final_maps['pred_2023_lh']   # либо 'magt_2023' — проверь
lats = final_maps['lats']
lons = final_maps['lons']

print(f"MAGT shape: {magt_2023.shape}, lats: {lats.shape}, lons: {lons.shape}")
print(f"MAGT диапазон: [{np.nanmin(magt_2023):.2f}, {np.nanmax(magt_2023):.2f}] °C")
print(f"MAGT медиана: {np.nanmedian(magt_2023):.2f} °C")

boreholes = load_borehole_data(BOREHOLES_CSV)
print(f"\nЗагружено бурений: {len(boreholes)}")
print(boreholes.head())

## 2. Сопоставляем предсказания с бурениями

In [ ]:
paired = pair_predictions_with_boreholes(magt_2023, boreholes, lats, lons)
print(f"Сопоставлено точек: {len(paired)} (из {len(boreholes)})")
print(paired[['lat', 'lon', 'magt_obs', 'magt_pred', 'zone_pred']].head(10))

err = paired['magt_pred'].values - paired['magt_obs'].values
baseline = {
    'rmse': np.sqrt(np.mean(err**2)),
    'bias': np.mean(err),
    'mae':  np.mean(np.abs(err)),
}
print(f"\nBASELINE (до коррекции):")
print(f"  RMSE: {baseline['rmse']:.3f} °C")
print(f"  Bias: {baseline['bias']:+.3f} °C   (отрицательный = модель холоднее)")
print(f"  MAE:  {baseline['mae']:.3f} °C")

## 3. Сравниваем 4 стратегии калибровки через k-fold CV

CV нужен, чтобы не оверфитнуть на 33 точках. Выбираем стратегию с минимальным
CV RMSE, но если две стратегии близки (<0.05°C) — берём более простую.

In [ ]:
results = compare_strategies(paired, n_folds=5, random_state=42)

print(f"{'Strategy':<12} {'CV RMSE':>14} {'CV Bias':>10} {'Train RMSE':>12} {'Train Bias':>12}")
print("-" * 70)
for name, r in results.items():
    print(f"{name:<12} "
          f"{r['cv']['rmse_mean']:>7.3f}±{r['cv']['rmse_std']:.2f} "
          f"{r['cv']['bias_mean']:>+10.3f} "
          f"{r['train']['rmse']:>12.3f} "
          f"{r['train']['bias']:>+12.3f}")

best_name = pick_best_strategy(results, prefer_simpler_when_close=0.05)
print(f"\nВыбрана стратегия: {best_name}")
best_corrector = results[best_name]['corrector']

In [ ]:
rows = []
for name, r in results.items():
    rows.append({
        'strategy': name,
        'cv_rmse_mean': r['cv']['rmse_mean'],
        'cv_rmse_std':  r['cv']['rmse_std'],
        'cv_bias_mean': r['cv']['bias_mean'],
        'cv_mae_mean':  r['cv']['mae_mean'],
        'train_rmse':   r['train']['rmse'],
        'train_bias':   r['train']['bias'],
        'is_chosen':    (name == best_name),
    })
summary_df = pd.DataFrame(rows)
summary_df.to_csv(METRICS_DIR / 'bias_correction_summary.csv', index=False)
print(summary_df.to_string(index=False))

## 4. Bootstrap для оценки неопределённости коэффициентов

Только для глобальной линейной — даёт цифры под защиту:
«коэффициент калибровки a = X ± σ_a».

In [ ]:
boot = bootstrap_corrector(paired, fit_global_linear, n_boot=2000)
print("Bootstrap (n=2000) для global linear:")
print(f"  slope     = {boot['slope_mean']:.3f} ± {boot['slope_std']:.3f}")
print(f"  intercept = {boot['intercept_mean']:+.3f} ± {boot['intercept_std']:.3f}")
print(f"  Успешных ресэмплов: {boot['n_successful_resamples']}/2000")

## 5. Применяем калибровку к полной карте

In [ ]:
# Если у нас есть карта зон мерзлоты — используем её для per_zone
zones_path = MAPS_DIR / 'zones_2023.npz'
if zones_path.exists():
    zones_map = np.load(zones_path)['zones']
    print(f"Загружена карта зон: {zones_map.shape}, уникальные значения: {np.unique(zones_map)}")
else:
    zones_map = None
    print("Карта зон не найдена — будем считать зоны по pred")

magt_corrected = apply_correction(magt_2023, best_corrector, zones=zones_map)

print(f"\nMAGT до:    [{np.nanmin(magt_2023):.2f}, {np.nanmax(magt_2023):.2f}] °C, медиана {np.nanmedian(magt_2023):.2f}")
print(f"MAGT после: [{np.nanmin(magt_corrected):.2f}, {np.nanmax(magt_corrected):.2f}] °C, медиана {np.nanmedian(magt_corrected):.2f}")
print(f"Δ медиана: {np.nanmedian(magt_corrected) - np.nanmedian(magt_2023):+.3f} °C")

## 6. Финальные метрики после калибровки

In [ ]:
paired_after = pair_predictions_with_boreholes(magt_corrected, boreholes, lats, lons)
err_after = paired_after['magt_pred'].values - paired_after['magt_obs'].values
after = {
    'rmse': np.sqrt(np.mean(err_after**2)),
    'bias': np.mean(err_after),
    'mae':  np.mean(np.abs(err_after)),
}

print("СРАВНЕНИЕ — против 33 бурений:")
print(f"  {'':<10} {'BEFORE':>10} {'AFTER':>10} {'Δ':>10}")
print(f"  {'RMSE':<10} {baseline['rmse']:>10.3f} {after['rmse']:>10.3f} {after['rmse']-baseline['rmse']:>+10.3f}")
print(f"  {'Bias':<10} {baseline['bias']:>+10.3f} {after['bias']:>+10.3f} {after['bias']-baseline['bias']:>+10.3f}")
print(f"  {'MAE':<10} {baseline['mae']:>10.3f} {after['mae']:>10.3f} {after['mae']-baseline['mae']:>+10.3f}")

print("\nAcceptance:")
print("  + RMSE упал" if after['rmse'] < baseline['rmse'] else "  - RMSE НЕ упал — пересмотри стратегию")
print(f"  + |Bias| < 0.3" if abs(after['bias']) < 0.3 else f"  - |Bias|={abs(after['bias']):.2f} >= 0.3")

## 7. Визуализация: scatter до/после

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, df, title, metrics in [
    (axes[0], paired, 'ДО калибровки', baseline),
    (axes[1], paired_after, 'ПОСЛЕ калибровки', after),
]:
    ax.scatter(df['magt_obs'], df['magt_pred'], alpha=0.7, s=40, edgecolor='k', linewidth=0.5)
    lim = [min(df['magt_obs'].min(), df['magt_pred'].min()) - 1,
           max(df['magt_obs'].max(), df['magt_pred'].max()) + 1]
    ax.plot(lim, lim, 'r--', label='y=x', alpha=0.7)
    ax.set_xlabel('MAGT observed, °C (бурения)')
    ax.set_ylabel('MAGT predicted, °C')
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_title(f"{title}\nRMSE={metrics['rmse']:.2f}, bias={metrics['bias']:+.2f}, n={len(df)}")
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'bias_correction_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Визуализация: карта до/после

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 12))

im0 = axes[0].pcolormesh(lons, lats, magt_2023, cmap='RdBu_r', vmin=-12, vmax=2, shading='auto')
axes[0].set_title('MAGT 2023 — ДО калибровки', fontsize=14)
axes[0].set_xlabel('Долгота, °E'); axes[0].set_ylabel('Широта, °N')
plt.colorbar(im0, ax=axes[0], label='MAGT, °C', fraction=0.025)

im1 = axes[1].pcolormesh(lons, lats, magt_corrected, cmap='RdBu_r', vmin=-12, vmax=2, shading='auto')
axes[1].set_title(f'MAGT 2023 — ПОСЛЕ калибровки (стратегия: {best_name})', fontsize=14)
axes[1].set_xlabel('Долгота, °E'); axes[1].set_ylabel('Широта, °N')
plt.colorbar(im1, ax=axes[1], label='MAGT, °C', fraction=0.025)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'bias_correction_before_after_map.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Сохраняем скорректированную карту

In [ ]:
out_path = MAPS_DIR / 'MAGT_2023_bias_corrected.npz'
np.savez_compressed(
    out_path,
    magt_2023_corrected=magt_corrected,
    magt_2023_raw=magt_2023,
    lats=lats,
    lons=lons,
    strategy=best_name,
    rmse_before=baseline['rmse'],
    rmse_after=after['rmse'],
    bias_before=baseline['bias'],
    bias_after=after['bias'],
)
print(f"Сохранено: {out_path}")
print(f"Размер: {out_path.stat().st_size / 1e6:.2f} MB")

## 10. Sanity-check: сравнение с Obu 2019

Ожидание: RMSE против Obu может ВЫРАСТИ — это НОРМАЛЬНО.
Если против бурений RMSE упал, а против Obu вырос — мы убрали унаследованный
от Obu холодный bias, что и было целью.

In [ ]:
obu_path = BASE_DIR / 'data' / 'obu_2019_reprojected.npz'
if obu_path.exists():
    obu = np.load(obu_path)['obu_on_grid']
    mask = ~(np.isnan(obu) | np.isnan(magt_2023) | np.isnan(magt_corrected))

    err_before_obu = (magt_2023[mask] - obu[mask])
    err_after_obu  = (magt_corrected[mask] - obu[mask])

    print("Против Obu 2019:")
    print(f"  RMSE до:    {np.sqrt(np.mean(err_before_obu**2)):.3f}")
    print(f"  RMSE после: {np.sqrt(np.mean(err_after_obu**2)):.3f}")
    print(f"  Bias до:    {np.mean(err_before_obu):+.3f}")
    print(f"  Bias после: {np.mean(err_after_obu):+.3f}")
    print("\nЕсли RMSE против Obu вырос, а против бурений упал — это ОЖИДАЕМО")
else:
    print(f"Карта Obu не найдена в {obu_path} — пропускаем sanity-check")

## 11. Что добавить в Главу 5 отчёта

```
Для устранения cold bias, унаследованного от MODIS LST (Westermann 2012,
остаточный bias ~0.8°C после ERA gap-filling) и от схемы расчёта nf-фактора
в континентальной Сибири (Obu 2019, занижение MAGT 1-6°C в районе Якутска),
применена post-hoc эмпирическая калибровка против 33 чистых бурений.
Сравнили 4 стратегии калибровки через 5-fold cross-validation:
identity, глобальная линейная, кусочно-линейная и зональная.
Выбрана стратегия {best_name} по правилу Оккама — простейшая в пределах 0.05°C
от минимального CV RMSE. После калибровки: RMSE против бурений снизился
с X.XX до Y.YY°C, bias — с +X.XX до +Y.YY°C.
```

Подставь X.XX/Y.YY из ячейки 11.